In [1]:
import sqlite3
import pandas as pd

In [2]:
df = pd.read_csv("../data/cleaned_loans.csv")

In [3]:
df["is_default"] = df["loan_status"].apply(
    lambda x: 0 if x == "Fully Paid" else 1
)

In [4]:
df.rename(columns={"default": "is_default"}, inplace=True)

In [5]:
conn = sqlite3.connect("../data/loan_portfolio.db")

In [6]:
df.to_sql(
    "loans",
    conn,
    if_exists="replace",
    index=False
)

1345350

In [7]:
def run_query(query):
    return pd.read_sql(query, conn)

# SECTION A - Portfolio Overview

### 1: How many loans are in the portfolio?

In [8]:
run_query("""
SELECT COUNT(*) AS Total_Loans
FROM loans
""")

,Total_Loans
0,1345350


### 2: What is the total portfolio value?

In [9]:
def print_kpi(query, column_name, prefix=""):
    result = pd.read_sql(query, conn)
    value = result.loc[0, column_name]
    print(f"{prefix}{value:,.2f}")

In [10]:
print_kpi("""
SELECT
SUM(loan_amnt) AS Portfolio_Value
FROM loans;
""", "Portfolio_Value", "$")

$19,399,906,575.00


### 3: What is the average loan amount?

In [11]:
run_query("""SELECT

ROUND(
AVG(loan_amnt),
2
)

AS Average_Loan

FROM loans;""")

,Average_Loan
0,14419.97


### 4: What is the overall default rate?

In [12]:
query = """
SELECT
ROUND(
AVG(is_default) * 100,
2
) AS Default_Rate
FROM loans;
"""

pd.read_sql(query, conn)

,Default_Rate
0,19.96


#### Business Insight
* The portfolio contains over 1.3 million historical loans.
* Total lending exposure exceeds $19 billion.
* The default rate provides the baseline portfolio risk that later analyses will explain.

# SECTION B - Borrower Risk Analysis 

### 5: Which loan grades have the highest default rates?

In [13]:
query = """
SELECT
    grade,
    COUNT(*) AS Total_Loans,
    ROUND(AVG(is_default) * 100, 2) AS Default_Rate
FROM loans
GROUP BY grade
ORDER BY Default_Rate DESC;
"""

grade = pd.read_sql(query, conn)
grade

,grade,Total_Loans,Default_Rate
0,G,9132,49.93
1,F,32059,45.20
2,E,93656,38.48
3,D,200966,30.39
4,C,381694,22.44
5,B,392748,13.39
6,A,235095,6.04


### 6: Which loan purposes are the riskiest?

In [14]:
query = """
SELECT
    purpose,
    COUNT(*) AS Total_Loans,
    ROUND(AVG(is_default) * 100, 2) AS Default_Rate
FROM loans
GROUP BY purpose
ORDER BY Default_Rate DESC;
"""

purpose = pd.read_sql(query, conn)
purpose

,purpose,Total_Loans,Default_Rate
0,small_business,15416,29.71
1,renewable_energy,933,23.69
2,moving,9480,23.35
3,house,7254,21.89
4,medical,15556,21.79
5,debt_consolidation,780342,21.15
6,other,77877,21.04
7,vacation,9065,19.17
8,major_purchase,29427,18.61
9,home_improvement,87507,17.72


### 7: Which verification status performs best?

In [15]:
query = """
SELECT
    verification_status,
    COUNT(*) AS Total_Loans,
    ROUND(AVG(is_default) * 100, 2) AS Default_Rate
FROM loans
GROUP BY verification_status
ORDER BY Default_Rate;
"""

verification = pd.read_sql(query, conn)
verification

,verification_status,Total_Loans,Default_Rate
0,Not Verified,405709,14.68
1,Source Verified,521289,20.96
2,Verified,418352,23.86


### 8: Does home ownership affect default?

In [16]:
query = """
SELECT
    home_ownership,
    COUNT(*) AS Total_Loans,
    ROUND(AVG(is_default) * 100, 2) AS Default_Rate
FROM loans
GROUP BY home_ownership
ORDER BY Default_Rate DESC;
"""

home = pd.read_sql(query, conn)
home

,home_ownership,Total_Loans,Default_Rate
0,RENT,534436,23.22
1,OWN,144840,20.62
2,ANY,286,19.58
3,OTHER,144,18.75
4,MORTGAGE,665596,17.21
5,NONE,48,14.58


### 9: Does employment length influence default?

In [17]:
query = """
SELECT
    emp_length,
    COUNT(*) AS Total_Loans,
    ROUND(AVG(is_default) * 100, 2) AS Default_Rate
FROM loans
GROUP BY emp_length
ORDER BY emp_length;
"""

employment = pd.read_sql(query, conn)
employment

,emp_length,Total_Loans,Default_Rate
0,-1.0,628790,20.10
1,1.0,88495,20.57
2,2.0,121751,19.81
3,3.0,107602,19.97
4,4.0,80558,19.74
5,5.0,84154,19.60
6,6.0,62735,19.35
7,7.0,59624,19.49
8,8.0,60704,19.94
9,9.0,50937,19.90


# SECTION C — Credit Profile Analysis

### 10: Average FICO Score by Loan Status

In [18]:
query = """
SELECT
    loan_status,
    ROUND(AVG(fico),0) AS Average_FICO
FROM loans
GROUP BY loan_status;
"""

fico = pd.read_sql(query, conn)
fico

,loan_status,Average_FICO
0,Charged Off,690.0
1,Default,705.0
2,Fully Paid,700.0


### 11: Average Interest Rate by Loan Grade

In [19]:
query = """
SELECT
    grade,
    ROUND(AVG(int_rate),2) AS Average_Interest
FROM loans
GROUP BY grade
ORDER BY grade;
"""

interest = pd.read_sql(query, conn)
interest

,grade,Average_Interest
0,A,7.11
1,B,10.68
2,C,14.02
3,D,17.72
4,E,21.14
5,F,24.93
6,G,27.73


### 12: Average DTI by Loan Grade

In [20]:
query = """
SELECT
    grade,
    ROUND(AVG(dti),2) AS Average_DTI
FROM loans
GROUP BY grade
ORDER BY grade;
"""

dti = pd.read_sql(query, conn)
dti

,grade,Average_DTI
0,A,15.61
1,B,17.38
2,C,18.92
3,D,20.19
4,E,20.84
5,F,20.97
6,G,21.51


### 13: Average Annual Income by Loan Grade

In [21]:
query = """
SELECT
    grade,
    ROUND(AVG(annual_inc),2) AS Average_Income
FROM loans
GROUP BY grade
ORDER BY grade;
"""

income = pd.read_sql(query, conn)

income["Average_Income"] = income["Average_Income"].map(
    lambda x: f"${x:,.2f}"
)

income

,grade,Average_Income
0,A,"$88,958.06"
1,B,"$76,468.56"
2,C,"$72,703.21"
3,D,"$70,156.79"
4,E,"$72,022.73"
5,F,"$73,069.91"
6,G,"$76,192.42"


# SECTION D — Portfolio Performance

### 14: Monthly Loan Issuance Trend

In [22]:
query = """
SELECT
    substr(issue_d,1,7) AS Month,
    COUNT(*) AS Loans_Issued
FROM loans
GROUP BY Month
ORDER BY Month;
"""

monthly_loans = pd.read_sql(query, conn)
monthly_loans

,Month,Loans_Issued
0,2007-06,1
1,2007-07,30
2,2007-08,33
3,2007-09,18
4,2007-10,47
...,...,...
134,2018-08,3569
135,2018-09,2423
136,2018-10,2150
137,2018-11,1632


### 15: Monthly Lending Volume

In [23]:
query = """
SELECT
    substr(issue_d,1,7) AS Month,
    SUM(loan_amnt) AS Portfolio_Value
FROM loans
GROUP BY Month
ORDER BY Month;
"""

portfolio = pd.read_sql(query, conn)

portfolio["Portfolio_Value"] = portfolio["Portfolio_Value"].map(
    lambda x: f"${x:,.0f}"
)

portfolio

,Month,Portfolio_Value
0,2007-06,"$7,500"
1,2007-07,"$171,700"
2,2007-08,"$208,475"
3,2007-09,"$146,025"
4,2007-10,"$318,775"
...,...,...
134,2018-08,"$50,566,950"
135,2018-09,"$35,261,800"
136,2018-10,"$30,133,225"
137,2018-11,"$22,599,600"


### 16: Do Longer Loan Terms Have Higher Default Rates?

In [24]:
query = """
SELECT
    term,
    COUNT(*) AS Total_Loans,
    ROUND(AVG(is_default) * 100,2) AS Default_Rate
FROM loans
GROUP BY term;
"""

term = pd.read_sql(query, conn)
term

,term,Total_Loans,Default_Rate
0,36.0,1020768,16.00
1,60.0,324582,32.45


# SECTION E — Risk Segmentation

### 17: High-Risk Borrower Profile

In [25]:
query = """
SELECT
    grade,
    purpose,
    ROUND(AVG(fico),0) AS Avg_FICO,
    ROUND(AVG(dti),2) AS Avg_DTI,
    ROUND(AVG(int_rate),2) AS Avg_Interest,
    ROUND(AVG(is_default) * 100,2) AS Default_Rate
FROM loans
GROUP BY grade, purpose
HAVING COUNT(*) > 100
ORDER BY Default_Rate DESC
LIMIT 20;
"""

high_risk = pd.read_sql(query, conn)
high_risk

,grade,purpose,Avg_FICO,Avg_DTI,Avg_Interest,Default_Rate
0,G,debt_consolidation,679.0,23.07,27.86,51.44
1,G,small_business,686.0,15.64,26.61,50.72
2,G,moving,681.0,17.38,27.35,49.15
3,G,other,684.0,19.07,27.65,48.83
4,G,credit_card,678.0,23.80,27.89,48.17
5,G,medical,682.0,19.75,27.19,47.71
6,F,major_purchase,689.0,16.80,25.21,47.59
7,G,major_purchase,689.0,18.76,27.97,47.45
8,F,debt_consolidation,681.0,22.20,24.95,47.40
9,F,credit_card,679.0,22.99,25.05,43.78


### 18: Top 10 Largest Loans

In [26]:
query = """
SELECT
    loan_amnt,
    grade,
    purpose,
    annual_inc,
    fico,
    dti,
    int_rate
FROM loans
ORDER BY loan_amnt DESC
LIMIT 10;
"""

largest_loans = pd.read_sql(query, conn)
largest_loans

,loan_amnt,grade,purpose,annual_inc,fico,dti,int_rate
0,40000.0,B,debt_consolidation,150000.0,732.0,4.09,9.43
1,40000.0,B,debt_consolidation,275000.0,722.0,15.55,9.43
2,40000.0,B,debt_consolidation,130000.0,722.0,8.66,9.43
3,40000.0,B,home_improvement,94000.0,682.0,2.32,9.92
4,40000.0,B,debt_consolidation,52000.0,732.0,24.65,10.90
5,40000.0,A,major_purchase,75000.0,777.0,18.78,7.96
6,40000.0,B,debt_consolidation,140000.0,702.0,7.89,9.92
7,40000.0,B,other,160000.0,687.0,2.16,9.92
8,40000.0,C,debt_consolidation,0.0,757.0,17.84,15.04
9,40000.0,A,home_improvement,139000.0,812.0,3.73,5.31


### 19: Average Revolving Credit Utilization by Loan Status

In [27]:
query = """
SELECT
    loan_status,
    ROUND(AVG(revol_util),2) AS Average_Utilization
FROM loans
GROUP BY loan_status;
"""

utilization = pd.read_sql(query, conn)
utilization

,loan_status,Average_Utilization
0,Charged Off,54.76
1,Default,43.59
2,Fully Paid,51.07


### 20: Executive Portfolio Summary by Loan Grade

In [28]:
query = """
SELECT
    grade,
    COUNT(*) AS Total_Loans,
    ROUND(SUM(loan_amnt),2) AS Portfolio_Value,
    ROUND(AVG(is_default) * 100,2) AS Default_Rate,
    ROUND(AVG(int_rate),2) AS Average_Interest,
    ROUND(AVG(fico),0) AS Average_FICO,
    ROUND(AVG(dti),2) AS Average_DTI,
    ROUND(AVG(annual_inc),2) AS Average_Income
FROM loans
GROUP BY grade
ORDER BY grade;
"""

summary = pd.read_sql(query, conn)

summary["Portfolio_Value"] = summary["Portfolio_Value"].map(
    lambda x: f"${x:,.0f}"
)

summary["Average_Income"] = summary["Average_Income"].map(
    lambda x: f"${x:,.0f}"
)

summary

,grade,Total_Loans,Portfolio_Value,Default_Rate,Average_Interest,Average_FICO,Average_DTI,Average_Income
0,A,235095,"$3,266,020,650",6.04,7.11,730.0,15.61,"$88,958"
1,B,392748,"$5,199,049,650",13.39,10.68,699.0,17.38,"$76,469"
2,C,381694,"$5,415,625,225",22.44,14.02,690.0,18.92,"$72,703"
3,D,200966,"$3,069,206,975",30.39,17.72,685.0,20.19,"$70,157"
4,E,93656,"$1,650,055,125",38.48,21.14,684.0,20.84,"$72,023"
5,F,32059,"$611,931,975",45.20,24.93,682.0,20.97,"$73,070"
6,G,9132,"$188,016,975",49.93,27.73,681.0,21.51,"$76,192"


## How is the portfolio distributed across DTI risk bands?

In [29]:
query = """
SELECT

CASE
    WHEN dti < 10 THEN 'Very Low Risk'
    WHEN dti < 20 THEN 'Low Risk'
    WHEN dti < 30 THEN 'Moderate Risk'
    WHEN dti < 40 THEN 'High Risk'
    ELSE 'Very High Risk'
END AS DTI_Risk,

COUNT(*) AS Total_Loans,

ROUND(AVG(is_default) * 100,2) AS Default_Rate

FROM loans

GROUP BY DTI_Risk

ORDER BY Default_Rate;
"""

dti_risk = pd.read_sql(query, conn)

dti_risk

,DTI_Risk,Total_Loans,Default_Rate
0,Very Low Risk,245723,14.89
1,Low Risk,562295,17.84
2,Moderate Risk,408647,23.04
3,High Risk,121915,29.09
4,Very High Risk,6770,30.55


## Which loan grades are the riskiest?

In [30]:
query = """
SELECT

grade,

COUNT(*) AS Loans,

ROUND(AVG(is_default) * 100,2) AS Default_Rate,

RANK() OVER(
ORDER BY AVG(is_default) DESC
) AS Risk_Rank

FROM loans

GROUP BY grade;
"""

grade_rank = pd.read_sql(query, conn)

grade_rank

,grade,Loans,Default_Rate,Risk_Rank
0,G,9132,49.93,1
1,F,32059,45.20,2
2,E,93656,38.48,3
3,D,200966,30.39,4
4,C,381694,22.44,5
5,B,392748,13.39,6
6,A,235095,6.04,7


## Can we segment borrowers into equally sized FICO groups?

In [31]:
query = """
SELECT

fico,

annual_inc,

loan_amnt,

NTILE(5) OVER(
ORDER BY fico
) AS FICO_Quintile

FROM loans;
"""

fico_quintile = pd.read_sql(query, conn)

fico_quintile.head()

,fico,annual_inc,loan_amnt,FICO_Quintile
0,627.0,52200.0,2700.0,1
1,632.0,25000.0,1500.0,1
2,662.0,75000.0,24250.0,1
3,662.0,110000.0,23100.0,1
4,662.0,65000.0,27300.0,1


## How has loan issuance changed over time?

In [32]:
query = """
WITH monthly AS (

SELECT

substr(issue_d,1,7) AS Month,

COUNT(*) AS Loans

FROM loans

GROUP BY Month

)

SELECT

Month,

Loans,

LAG(Loans) OVER(
ORDER BY Month
) AS Previous_Month,

Loans -
LAG(Loans) OVER(
ORDER BY Month
) AS Loan_Growth

FROM monthly;
"""

growth = pd.read_sql(query, conn)

growth

,Month,Loans,Previous_Month,Loan_Growth
0,2007-06,1,NaN,NaN
1,2007-07,30,1.0,29.0
2,2007-08,33,30.0,3.0
3,2007-09,18,33.0,-15.0
4,2007-10,47,18.0,29.0
...,...,...,...,...
134,2018-08,3569,4317.0,-748.0
135,2018-09,2423,3569.0,-1146.0
136,2018-10,2150,2423.0,-273.0
137,2018-11,1632,2150.0,-518.0


## How has the cumulative portfolio value grown over time?

In [33]:
query = """
WITH monthly AS (

SELECT

substr(issue_d,1,7) AS Month,

SUM(loan_amnt) AS Portfolio

FROM loans

GROUP BY Month

)

SELECT

Month,

Portfolio,

SUM(Portfolio)

OVER(

ORDER BY Month

ROWS BETWEEN UNBOUNDED PRECEDING

AND CURRENT ROW

)

AS Cumulative_Portfolio

FROM monthly;
"""

portfolio_growth = pd.read_sql(query, conn)

portfolio_growth

,Month,Portfolio,Cumulative_Portfolio
0,2007-06,7500.0,7.500000e+03
1,2007-07,171700.0,1.792000e+05
2,2007-08,208475.0,3.876750e+05
3,2007-09,146025.0,5.337000e+05
4,2007-10,318775.0,8.524750e+05
...,...,...,...
134,2018-08,50566950.0,1.929461e+10
135,2018-09,35261800.0,1.932988e+10
136,2018-10,30133225.0,1.936001e+10
137,2018-11,22599600.0,1.938261e+10


## Which combinations of loan grade, purpose, and home ownership have the highest default rates?

In [34]:
query = """
SELECT

grade,

purpose,

home_ownership,

COUNT(*) AS Loans,

ROUND(AVG(is_default) * 100,2) AS Default_Rate

FROM loans

GROUP BY

grade,
purpose,
home_ownership

HAVING COUNT(*) > 100

ORDER BY Default_Rate DESC

LIMIT 10;
"""

segments = pd.read_sql(query, conn)

segments

,grade,purpose,home_ownership,Loans,Default_Rate
0,G,debt_consolidation,RENT,2375,57.14
1,G,debt_consolidation,OWN,604,52.98
2,G,credit_card,RENT,243,52.67
3,G,small_business,RENT,235,52.34
4,F,major_purchase,RENT,352,51.99
5,F,debt_consolidation,RENT,8901,51.75
6,G,other,RENT,504,51.59
7,G,major_purchase,RENT,114,50.88
8,G,small_business,MORTGAGE,187,49.73
9,F,debt_consolidation,OWN,2030,48.87
